In [ ]:
import mlflow
import pandas as pd

mlflow.set_tracking_uri("sqlite:///../mlflow.db")
mlflow.set_experiment("House Price Prediction Experiment")

In [3]:
house_sales = pd.read_csv("../house_sales.csv")

In [6]:
house_sales.describe()

,Rooms,Price,Distance,Postcode,Bedroom2,Bathroom,Car,Landsize,BuildingArea,YearBuilt,Lattitude,Longtitude,Propertycount
count,13580.000000,1.358000e+04,13580.000000,13580.000000,13580.000000,13580.000000,13518.000000,13580.000000,7130.000000,8205.000000,13580.000000,13580.000000,13580.000000
mean,2.937997,1.075684e+06,10.137776,3105.301915,2.914728,1.534242,1.610075,558.416127,151.967650,1964.684217,-37.809203,144.995216,7454.417378
std,0.955748,6.393107e+05,5.868725,90.676964,0.965921,0.691712,0.962634,3990.669241,541.014538,37.273762,0.079260,0.103916,4378.581772
min,1.000000,8.500000e+04,0.000000,3000.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1196.000000,-38.182550,144.431810,249.000000
25%,2.000000,6.500000e+05,6.100000,3044.000000,2.000000,1.000000,1.000000,177.000000,93.000000,1940.000000,-37.856822,144.929600,4380.000000
50%,3.000000,9.030000e+05,9.200000,3084.000000,3.000000,1.000000,2.000000,440.000000,126.000000,1970.000000,-37.802355,145.000100,6555.000000
75%,3.000000,1.330000e+06,13.000000,3148.000000,3.000000,2.000000,2.000000,651.000000,174.000000,1999.000000,-37.756400,145.058305,10331.000000
max,10.000000,9.000000e+06,48.100000,3977.000000,20.000000,8.000000,10.000000,433014.000000,44515.000000,2018.000000,-37.408530,145.526350,21650.000000


In [8]:
house_sales = house_sales.dropna(axis=0)
house_sales.describe()

,Rooms,Price,Distance,Postcode,Bedroom2,Bathroom,Car,Landsize,BuildingArea,YearBuilt,Lattitude,Longtitude,Propertycount
count,6196.000000,6.196000e+03,6196.000000,6196.000000,6196.000000,6196.000000,6196.000000,6196.000000,6196.000000,6196.000000,6196.000000,6196.000000,6196.000000
mean,2.931407,1.068828e+06,9.751097,3101.947708,2.902034,1.576340,1.573596,471.006940,141.568645,1964.081988,-37.807904,144.990201,7435.489509
std,0.971079,6.751564e+05,5.612065,86.421604,0.970055,0.711362,0.929947,897.449881,90.834824,38.105673,0.075850,0.099165,4337.698917
min,1.000000,1.310000e+05,0.000000,3000.000000,0.000000,1.000000,0.000000,0.000000,0.000000,1196.000000,-38.164920,144.542370,389.000000
25%,2.000000,6.200000e+05,5.900000,3044.000000,2.000000,1.000000,1.000000,152.000000,91.000000,1940.000000,-37.855438,144.926198,4383.750000
50%,3.000000,8.800000e+05,9.000000,3081.000000,3.000000,1.000000,1.000000,373.000000,124.000000,1970.000000,-37.802250,144.995800,6567.000000
75%,4.000000,1.325000e+06,12.400000,3147.000000,3.000000,2.000000,2.000000,628.000000,170.000000,2000.000000,-37.758200,145.052700,10175.000000
max,8.000000,9.000000e+06,47.400000,3977.000000,9.000000,8.000000,10.000000,37000.000000,3112.000000,2018.000000,-37.457090,145.526350,21650.000000


In [ ]:
features = ['Rooms', 'Bathroom', 'Landsize', 'BuildingArea', 'YearBuilt', 'Lattitude', 'Longtitude']
X = house_sales[features]
y = house_sales["Price"]
print(X.shape)
print(y.shape)

In [20]:
from sklearn.model_selection import train_test_split

# Split the data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=1)

In [39]:
# Fist Model
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

with mlflow.start_run(run_name = "Linear Regression Model"):
    lr_model_house_sales = LinearRegression()
    lr_model_house_sales.fit(X_train, y_train)

    y_preds_lr = lr_model_house_sales.predict(X_test)
    mae_lr = mean_absolute_error(y_test, y_preds_lr)
    mlflow.log_metric("mean_absolute_error", mae_lr)
    mlflow.sklearn.log_model(sk_model=lr_model_house_sales, name="lr_model_house_sales")
    print(f"Mean Absolute Error (Linear Regression Model): {mae_lr}")

2026/06/10 11:15:25 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Mean Absolute Error (Linear Regression Model): 293237.74374876515


In [40]:
# Second Model
from sklearn.tree import DecisionTreeRegressor

with mlflow.start_run(run_name = "Decision Tree Regressor Model"):
    dt_model_house_sales = DecisionTreeRegressor(random_state=0)
    dt_model_house_sales.fit(X_train, y_train)

    y_preds_dt = dt_model_house_sales.predict(X_test)
    mae_dt = mean_absolute_error(y_test, y_preds_dt)
    mlflow.log_metric("mean_absolute_error", mae_dt)
    mlflow.sklearn.log_model(sk_model=dt_model_house_sales, name="dt_model_house_sales")
    print(f"Mean Absolute Error (Decision Tree Regressor Model): {mae_dt}")

2026/06/10 11:18:06 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Mean Absolute Error (Decision Tree Regressor Model): 248510.6655907037


In [ ]:
# Try different max_leaf_nodes (Underfitting and Overfitting)

In [41]:
def get_mae(max_leaf_nodes, X_train, X_test, y_train, y_test):
    with mlflow.start_run(run_name = f"Decision Tree Regressor Model - Max leaf nodes: {max_leaf_nodes}"):
        model = DecisionTreeRegressor(max_leaf_nodes=max_leaf_nodes, random_state=0)
        model.fit(X_train, y_train)
        y_preds = model.predict(X_test)
        mae = mean_absolute_error(y_test, y_preds)
        mlflow.log_metric("mean_absolute_error", mae)
        mlflow.sklearn.log_model(sk_model=model, name=f"dt_model_house_sales_max_leaf_nodes_{max_leaf_nodes}")
    return mae

In [42]:
# compare MAE with differing values of max_leaf_nodes
for max_leaf_nodes in [5, 50, 500, 5000]:
    mae = get_mae(max_leaf_nodes, X_train, X_test, y_train, y_test)
    print(f"Max leaf nodes: {max_leaf_nodes} \t\t Mean Absolute Error: {mae}")

2026/06/10 11:20:08 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/06/10 11:20:09 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Max leaf nodes: 5 		 Mean Absolute Error: 324110.91454760684


2026/06/10 11:20:12 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Max leaf nodes: 50 		 Mean Absolute Error: 252108.93840069315


2026/06/10 11:20:15 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Max leaf nodes: 500 		 Mean Absolute Error: 239204.22623333166
Max leaf nodes: 5000 		 Mean Absolute Error: 249358.76436410588


In [43]:
# Third Model
from sklearn.ensemble import RandomForestRegressor

with mlflow.start_run(run_name = "Random Forest Regressor"):
    rf_model_house_sales = RandomForestRegressor()
    rf_model_house_sales.fit(X_train, y_train)

    y_preds_rf = rf_model_house_sales.predict(X_test)
    mae_rf = mean_absolute_error(y_test, y_preds_rf)
    mlflow.log_metric("mean_absolute_error", mae_rf)
    mlflow.sklearn.log_model(sk_model=rf_model_house_sales, name="rf_model_house_sales")
    print(f"Mean Absolute Error (Random Forest Model): {mae_rf}")

2026/06/10 11:21:06 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Mean Absolute Error (Random Forest Model): 173221.17045559347


In [45]:
# Fit Model Using All Data
with mlflow.start_run(run_name = "Final Model"):
    final_model = RandomForestRegressor()
    final_model.fit(X, y)
    final_model_info = mlflow.sklearn.log_model(sk_model=final_model, name="final_model")

2026/06/10 12:12:23 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


In [ ]:
# mlflow models serve -m "<model-uri>" -p 5000
# Replace <model-uri> with your model's URI (e.g., runs:/<run_id>/model or models:/<model_name>/<version>). 
# This starts a local server at http://localhost:5000, exposing a REST API for inference requests. 
# For more details, see the official documentation at MLflow Model Serving and MLflow Model Deployment.
# Would you like more detail or example code for making predictions?

# https://mlflow.org/docs/latest/genai/serving/
# https://mlflow.org/docs/latest/ml/deployment/

# https://mlflow.org/docs/latest/ml/deployment/

In [46]:
final_model_info.model_uri

'models:/m-72fd213672a142f78e0d562f8751006a'

In [ ]:
# mlflow models serve -m runs:/<run_id>/model -p 5000
mlflow models serve -m models:/m-72fd213672a142f78e0d562f8751006a -p 5000

# You get this error:

2026/06/10 12:35:19 INFO mlflow.models.flavor_backend_registry: Selected backend for flavor 'python_function'
Traceback (most recent call last):
  File "/home/zelitosaide/Documents/mlops/.venv/bin/mlflow", line 8, in <module>
    sys.exit(cli())
             ~~~^^
  File "/home/zelitosaide/Documents/mlops/.venv/lib/python3.14/site-packages/click/core.py", line 1524, in __call__
    return self.main(*args, **kwargs)
           ~~~~~~~~~^^^^^^^^^^^^^^^^^
  File "/home/zelitosaide/Documents/mlops/.venv/lib/python3.14/site-packages/click/core.py", line 1445, in main
    rv = self.invoke(ctx)
  File "/home/zelitosaide/Documents/mlops/.venv/lib/python3.14/site-packages/click/core.py", line 1912, in invoke
    return _process_result(sub_ctx.command.invoke(sub_ctx))
                           ~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^
  File "/home/zelitosaide/Documents/mlops/.venv/lib/python3.14/site-packages/click/core.py", line 1912, in invoke
    return _process_result(sub_ctx.command.invoke(sub_ctx))
                           ~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^
  File "/home/zelitosaide/Documents/mlops/.venv/lib/python3.14/site-packages/click/core.py", line 1308, in invoke
    return ctx.invoke(self.callback, **ctx.params)
           ~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/zelitosaide/Documents/mlops/.venv/lib/python3.14/site-packages/click/core.py", line 877, in invoke
    return callback(*args, **kwargs)
  File "/home/zelitosaide/Documents/mlops/.venv/lib/python3.14/site-packages/mlflow/models/cli.py", line 104, in serve
    ).serve(model_uri=model_uri, port=port, host=host, timeout=timeout)
      ~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/zelitosaide/Documents/mlops/.venv/lib/python3.14/site-packages/mlflow/pyfunc/backend.py", line 301, in serve
    return self.prepare_env(local_path).execute(
           ~~~~~~~~~~~~~~~~^^^^^^^^^^^^
  File "/home/zelitosaide/Documents/mlops/.venv/lib/python3.14/site-packages/mlflow/pyfunc/backend.py", line 141, in prepare_env
    activate_cmd = _get_or_create_virtualenv(
        local_path,
    ...<5 lines>...
        extra_envs=extra_envs,
    )
  File "/home/zelitosaide/Documents/mlops/.venv/lib/python3.14/site-packages/mlflow/utils/virtualenv.py", line 413, in _get_or_create_virtualenv
    _validate_pyenv_is_available()
    ~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^
  File "/home/zelitosaide/Documents/mlops/.venv/lib/python3.14/site-packages/mlflow/utils/virtualenv.py", line 63, in _validate_pyenv_is_available
    raise MlflowException(
        f"Could not find the pyenv binary. See {url} for installation instructions."
    )
mlflow.exceptions.MlflowException: Could not find the pyenv binary. See https://github.com/pyenv/pyenv#installation for installation instructions.

O erro ocorre porque o MLflow precisa do utilitário pyenv para recriar o ambiente virtual Python exato (com as versões das bibliotecas e dependências) que foi utilizado quando o modelo foi salvo. Como o pyenv não está instalado no seu sistema (Ubuntu/Linux), o comando mlflow models serve falha.Para resolver isso, você tem duas opções: instalar o pyenv (recomendado pelo MLflow) ou forçar o MLflow a usar o seu ambiente atual.

Solução 1: Ignorar o pyenv (Mais rápida)Se você já sabe que o seu ambiente virtual atual (.venv) possui todas as bibliotecas necessárias para rodar o modelo (como o scikit-learn, pandas, etc.), você pode dizer para o MLflow não tentar criar um ambiente novo.Basta adicionar a flag --env-manager local ao final do seu comando:

In [ ]:
# mlflow models serve -m models:/m-72fd213672a142f78e0d562f8751006a --env-manager local -p 5001

Solução 2: Instalar o pyenv (Solução definitiva)Se você quiser que o MLflow gerencie os ambientes de forma isolada e robusta, instale o pyenv seguindo estes passos no seu terminal:

1. Instale as dependências de compilação do Linux:
   ```bash
    sudo apt-get install -y make build-essential libssl-dev zlib1g-dev \
    libbz2-dev libreadline-dev libsqlite3-dev wget curl llvm libncurses-dev xz-utils \
    tk-dev libxml2-dev libxmlsec1-dev libffi-dev liblzma-dev
   ```
2. Instale o Pyenv:
   ```bash
    curl -fsSL https://pyenv.run | bash
   ```
3. Configure o seu terminal (Bash):Execute os comandos abaixo para que o sistema reconheça o pyenv:
   ```bash
    echo 'export PYENV_ROOT="$HOME/.pyenv"' >> ~/.bashrc
    echo '[[ -d $PYENV_ROOT/bin ]] && export PATH="$PYENV_ROOT/bin:$PATH"' >> ~/.bashrc
    echo 'eval "$(pyenv init - bash)"' >> ~/.bashrc
    echo 'eval "$(pyenv virtualenv-init -)"' >> ~/.bashrc
   ```
4. Reinicie o terminal:
   ```bash
   exec "$SHELL"
   ```

In [ ]:
curl -X POST http://127.0.0.1:5001/invocations   -H "Content-Type: application/json"   -d '{
      "dataframe_records": [
        {
          "Rooms": 2,
          "Bathroom": 1.0,
          "Landsize": 156.0,
          "BuildingArea": 79.0,
          "YearBuilt": 1900.0,
          "Lattitude": -37.8079,
          "Longtitude": 144.9934
        }
      ]
    }'

## build-docker
### Building a Docker image with --model-uri:

```bash
# Build a Docker image named 'my-image-name' that serves the model from run 'some-run-uuid'
# at run-relative artifact path 'my-model'
mlflow models build-docker --model-uri "runs:/some-run-uuid/my-model" --name "my-image-name"
# Serve the model
docker run -p 5001:8080 "my-image-name"
```

In [ ]:
mlflow models build-docker --model-uri models:/m-72fd213672a142f78e0d562f8751006a --name "house_sales_model"

# You will get this error:

mlflow models build-docker --model-uri models:/m-72fd213672a142f78e0d562f8751006a --name "house_sales_model"
2026/06/10 16:00:37 INFO mlflow.models.flavor_backend_registry: Selected backend for flavor 'python_function'
2026/06/10 16:00:37 INFO mlflow.pyfunc.backend: Building docker image with name house_sales_model
ERROR: permission denied while trying to connect to the docker API at unix:///var/run/docker.sock
Traceback (most recent call last):
  File "/home/zelitosaide/Documents/mlops/.venv/bin/mlflow", line 8, in <module>
    sys.exit(cli())
             ~~~^^
  File "/home/zelitosaide/Documents/mlops/.venv/lib/python3.14/site-packages/click/core.py", line 1524, in __call__
    return self.main(*args, **kwargs)
           ~~~~~~~~~^^^^^^^^^^^^^^^^^
  File "/home/zelitosaide/Documents/mlops/.venv/lib/python3.14/site-packages/click/core.py", line 1445, in main
    rv = self.invoke(ctx)
  File "/home/zelitosaide/Documents/mlops/.venv/lib/python3.14/site-packages/click/core.py", line 1912, in invoke
    return _process_result(sub_ctx.command.invoke(sub_ctx))
                           ~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^
  File "/home/zelitosaide/Documents/mlops/.venv/lib/python3.14/site-packages/click/core.py", line 1912, in invoke
    return _process_result(sub_ctx.command.invoke(sub_ctx))
                           ~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^
  File "/home/zelitosaide/Documents/mlops/.venv/lib/python3.14/site-packages/click/core.py", line 1308, in invoke
    return ctx.invoke(self.callback, **ctx.params)
           ~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/zelitosaide/Documents/mlops/.venv/lib/python3.14/site-packages/click/core.py", line 877, in invoke
    return callback(*args, **kwargs)
  File "/home/zelitosaide/Documents/mlops/.venv/lib/python3.14/site-packages/mlflow/models/cli.py", line 309, in build_docker
    python_api.build_docker(**kwargs)
    ~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^
  File "/home/zelitosaide/Documents/mlops/.venv/lib/python3.14/site-packages/mlflow/models/python_api.py", line 83, in build_docker
    get_flavor_backend(model_uri, docker_build=True, env_manager=env_manager).build_image(
    ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^
        model_uri,
        ^^^^^^^^^^
    ...<4 lines>...
        base_image=base_image,
        ^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/home/zelitosaide/Documents/mlops/.venv/lib/python3.14/site-packages/mlflow/pyfunc/backend.py", line 390, in build_image
    docker_utils.build_image_from_context(context_dir=cwd, image_name=image_name)
    ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/zelitosaide/Documents/mlops/.venv/lib/python3.14/site-packages/mlflow/models/docker_utils.py", line 228, in build_image_from_context
    raise RuntimeError("Docker build failed.")
RuntimeError: Docker build failed.

O erro permission denied ocorre porque o seu usuário atual do Ubuntu não tem permissão para acessar o serviço do Docker (docker.sock) sem usar o comando sudo.
Como o MLflow precisa rodar comandos do Docker internamente, ele falha imediatamente.

## Solução 1: Dar permissão ao seu usuário (Recomendada)

Esta é a solução definitiva. Ela adiciona o seu usuário ao grupo do Docker para que você possa rodar comandos do Docker diretamente sem precisar usar sudo.

1. Adicione o seu usuário ao grupo do Docker:

```bash
sudo usermod -aG docker $USER
```

2. Ative as novas configurações de grupo (evita ter que reiniciar a máquina):

```bash
newgrp docker
```

3. Reative o seu ambiente virtual (já que o comando acima abre uma nova sessão de terminal):
```bash
source .venv/bin/activate
```

4. Rode o comando do MLflow novamente:
```bash
mlflow models build-docker --model-uri models:/m-72fd213672a142f78e0d562f8751006a --name "house_sales_model"
```

## Solução 2: Rodar com sudo no ambiente virtual (Rápida)

Se você não quiser alterar as permissões de grupo agora, pode forçar a execução usando sudo. Contudo, para que o sudo reconheça o MLflow instalado dentro do seu ambiente virtual .venv, você precisa passar o caminho completo do executável:

```bash
sudo .venv/bin/mlflow models build-docker --model-uri models:/m-72fd213672a142f78e0d562f8751006a --name "house_sales_model"
```

In [ ]:
# I got this error

sudo .venv/bin/mlflow models build-docker --model-uri models:/m-72fd213672a142f78e0d562f8751006a --name "house_sales_model"
[sudo: authenticate] Password:              
2026/06/10 16:10:22 INFO mlflow.models.flavor_backend_registry: Selected backend for flavor 'python_function'
2026/06/10 16:10:22 INFO mlflow.pyfunc.backend: Building docker image with name house_sales_model
[+] Building 222.6s (9/16)                                                                                                                                                         docker:default
 => [internal] load build definition from Dockerfile                                                                                                                                         0.1s
 => => transferring dockerfile: 2.19kB                                                                                                                                                       0.0s
 => [internal] load metadata for docker.io/library/ubuntu:22.04                                                                                                                              4.0s
 => [internal] load .dockerignore                                                                                                                                                            0.0s
 => => transferring context: 2B                                                                                                                                                              0.0s
 => [ 1/12] FROM docker.io/library/ubuntu:22.04@sha256:4f838adc7181d9039ac795a7d0aba05a9bd9ecd480d294483169c5def983b64d                                                                      6.6s
 => => resolve docker.io/library/ubuntu:22.04@sha256:4f838adc7181d9039ac795a7d0aba05a9bd9ecd480d294483169c5def983b64d                                                                        0.0s
 => => sha256:40d16f30db405106ef8074779bdf41f012465c2a785bbeaa2eab9f2081099b47 29.74MB / 29.74MB                                                                                             4.6s
 => => extracting sha256:40d16f30db405106ef8074779bdf41f012465c2a785bbeaa2eab9f2081099b47                                                                                                    1.9s
 => [internal] load build context                                                                                                                                                            0.5s
 => => transferring context: 55.02MB                                                                                                                                                         0.4s
 => [ 2/12] RUN apt-get -y update && DEBIAN_FRONTEND=noninteractive TZ=Etc/UTC apt-get install -y --no-install-recommends wget curl nginx ca-certificates bzip2 build-essential cmake git-  72.6s
 => [ 3/12] RUN DEBIAN_FRONTEND=noninteractive TZ=Etc/UTC apt-get -y install tzdata     libssl-dev zlib1g-dev libbz2-dev libreadline-dev libsqlite3-dev wget curl llvm     libncursesw5-de  98.8s 
 => [ 4/12] RUN git clone     --depth 1     --branch $(git ls-remote --tags --sort=v:refname https://github.com/pyenv/pyenv.git | grep -o -E 'v[1-9]+(\.[1-9]+)+$' | tail -1)     https://g  6.3s 
 => ERROR [ 5/12] RUN apt install -y software-properties-common     && apt update     && add-apt-repository -y ppa:deadsnakes/ppa     && apt update     && apt install -y python3.10 pytho  34.0s 
------                                                                                                                                                                                            
 > [ 5/12] RUN apt install -y software-properties-common     && apt update     && add-apt-repository -y ppa:deadsnakes/ppa     && apt update     && apt install -y python3.10 python3.10-distutils     && apt remove -y python3-blinker     && ln -s -f $(which python3.10) /usr/bin/python     && wget https://bootstrap.pypa.io/get-pip.py -O /tmp/get-pip.py     && python /tmp/get-pip.py:      
0.327                                                                                                                                                                                             
0.327 WARNING: apt does not have a stable CLI interface. Use with caution in scripts.                                                                                                             
0.327                                                                                                                                                                                             
0.384 Reading package lists...
2.566 Building dependency tree...
3.059 Reading state information...
3.604 The following additional packages will be installed:
3.604   dbus dirmngr distro-info-data dmsetup gir1.2-glib-2.0
3.604   gir1.2-packagekitglib-1.0 gnupg gnupg-l10n gnupg-utils gpg gpg-agent
3.604   gpg-wks-client gpg-wks-server gpgconf gpgsm iso-codes libapparmor1
3.604   libappstream4 libargon2-1 libassuan0 libcryptsetup12 libdbus-1-3
3.604   libdevmapper1.02.1 libdw1 libgirepository-1.0-1 libglib2.0-bin
3.604   libgstreamer1.0-0 libip4tc2 libjson-c5 libkmod2 libksba8 libnpth0
3.604   libnss-systemd libpackagekit-glib2-18 libpam-systemd libpolkit-agent-1-0
3.605   libpolkit-gobject-1-0 libstemmer0d libsystemd0 libunwind8 libxmlb2
3.606   lsb-release networkd-dispatcher packagekit packagekit-tools pinentry-curses
3.606   pkexec policykit-1 polkitd python-apt-common python3-apt python3-blinker
3.606   python3-cffi-backend python3-cryptography python3-dbus python3-distro
3.606   python3-distro-info python3-gi python3-httplib2 python3-importlib-metadata
3.606   python3-jeepney python3-jwt python3-keyring python3-launchpadlib
3.606   python3-lazr.restfulclient python3-lazr.uri python3-more-itertools
3.606   python3-oauthlib python3-pyparsing python3-secretstorage python3-six
3.606   python3-software-properties python3-wadllib python3-zipp systemd
3.607   systemd-sysv systemd-timesyncd unattended-upgrades
3.609 Suggested packages:
3.611   default-dbus-session-bus | dbus-session-bus dbus-user-session
3.611   pinentry-gnome3 tor parcimonie xloadimage scdaemon isoquery
3.611   gstreamer1.0-tools iw | wireless-tools appstream pinentry-doc
3.611   python3-apt-dbg python-apt-doc python-blinker-doc python-cryptography-doc
3.611   python3-cryptography-vectors python-dbus-doc python3-crypto gir1.2-secret-1
3.611   gnome-keyring libkf5wallet-bin python3-keyrings.alt python3-testresources
3.611   python-pyparsing-doc python-secretstorage-doc systemd-container libfido2-1
3.611   libtss2-esys-3.0.2-0 libtss2-mu0 libtss2-rc0 bsd-mailx default-mta
3.611   | mail-transport-agent needrestart powermgmt-base
3.877 The following NEW packages will be installed:
3.877   dbus dirmngr distro-info-data dmsetup gir1.2-glib-2.0
3.877   gir1.2-packagekitglib-1.0 gnupg gnupg-l10n gnupg-utils gpg gpg-agent
3.877   gpg-wks-client gpg-wks-server gpgconf gpgsm iso-codes libapparmor1
3.877   libappstream4 libargon2-1 libassuan0 libcryptsetup12 libdbus-1-3
3.877   libdevmapper1.02.1 libdw1 libgirepository-1.0-1 libglib2.0-bin
3.878   libgstreamer1.0-0 libip4tc2 libjson-c5 libkmod2 libksba8 libnpth0
3.878   libnss-systemd libpackagekit-glib2-18 libpam-systemd libpolkit-agent-1-0
3.879   libpolkit-gobject-1-0 libstemmer0d libunwind8 libxmlb2 lsb-release
3.879   networkd-dispatcher packagekit packagekit-tools pinentry-curses pkexec
3.880   policykit-1 polkitd python-apt-common python3-apt python3-blinker
3.880   python3-cffi-backend python3-cryptography python3-dbus python3-distro
3.880   python3-distro-info python3-gi python3-httplib2 python3-importlib-metadata
3.880   python3-jeepney python3-jwt python3-keyring python3-launchpadlib
3.880   python3-lazr.restfulclient python3-lazr.uri python3-more-itertools
3.880   python3-oauthlib python3-pyparsing python3-secretstorage python3-six
3.880   python3-software-properties python3-wadllib python3-zipp
3.880   software-properties-common systemd systemd-sysv systemd-timesyncd
3.880   unattended-upgrades
3.883 The following packages will be upgraded:
3.884   libsystemd0
24.00 1 upgraded, 78 newly installed, 0 to remove and 1 not upgraded.
24.00 Need to get 16.4 MB of archives.
24.00 After this operation, 67.0 MB of additional disk space will be used.
24.00 Ign:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libsystemd0 amd64 249.11-0ubuntu3.21
24.00 Ign:2 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libapparmor1 amd64 3.0.4-2ubuntu2.5
24.00 Ign:3 http://archive.ubuntu.com/ubuntu jammy/main amd64 libargon2-1 amd64 0~20171227-0.3
24.00 Ign:4 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libdevmapper1.02.1 amd64 2:1.02.175-2.1ubuntu5
24.00 Ign:5 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libjson-c5 amd64 0.15-3~ubuntu1.22.04.2
24.00 Ign:6 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libcryptsetup12 amd64 2:2.4.3-1ubuntu1.3
24.00 Ign:7 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libip4tc2 amd64 1.8.7-1ubuntu5.2
24.00 Ign:8 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libkmod2 amd64 29-1ubuntu1.1
24.00 Ign:9 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 systemd amd64 249.11-0ubuntu3.21
24.00 Ign:10 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 systemd-sysv amd64 249.11-0ubuntu3.21
24.00 Ign:11 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libdbus-1-3 amd64 1.12.20-2ubuntu4.1
24.00 Ign:12 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 dbus amd64 1.12.20-2ubuntu4.1
24.00 Ign:13 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 distro-info-data all 0.52ubuntu0.12
24.00 Ign:14 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 dmsetup amd64 2:1.02.175-2.1ubuntu5
24.00 Ign:15 http://archive.ubuntu.com/ubuntu jammy/main amd64 libgirepository-1.0-1 amd64 1.72.0-1
24.00 Ign:16 http://archive.ubuntu.com/ubuntu jammy/main amd64 gir1.2-glib-2.0 amd64 1.72.0-1
24.00 Ign:17 http://archive.ubuntu.com/ubuntu jammy/main amd64 iso-codes all 4.9.0-1
24.00 Ign:18 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libnss-systemd amd64 249.11-0ubuntu3.21
24.00 Ign:19 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libpam-systemd amd64 249.11-0ubuntu3.21
24.00 Ign:20 http://archive.ubuntu.com/ubuntu jammy/main amd64 lsb-release all 11.1.0ubuntu4
24.00 Ign:21 http://archive.ubuntu.com/ubuntu jammy/main amd64 python3-dbus amd64 1.2.18-3build1
24.00 Ign:22 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 python3-gi amd64 3.42.1-0ubuntu1
24.00 Ign:23 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 networkd-dispatcher all 2.1-2ubuntu0.22.04.2
24.00 Ign:24 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 python-apt-common all 2.4.0ubuntu4.1
24.00 Ign:25 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 python3-apt amd64 2.4.0ubuntu4.1
24.00 Ign:26 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 systemd-timesyncd amd64 249.11-0ubuntu3.21
24.00 Ign:27 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 python3-distro-info all 1.1ubuntu0.2
24.00 Ign:28 http://archive.ubuntu.com/ubuntu jammy/main amd64 libassuan0 amd64 2.5.5-1build1
24.00 Ign:29 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 gpgconf amd64 2.2.27-3ubuntu2.5
24.00 Ign:30 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libksba8 amd64 1.6.0-2ubuntu0.2
24.00 Ign:31 http://archive.ubuntu.com/ubuntu jammy/main amd64 libnpth0 amd64 1.6-3build2
24.01 Ign:32 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 dirmngr amd64 2.2.27-3ubuntu2.5
24.01 Ign:33 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libpackagekit-glib2-18 amd64 1.2.5-2ubuntu3.1
24.01 Ign:34 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 gir1.2-packagekitglib-1.0 amd64 1.2.5-2ubuntu3.1
24.01 Ign:35 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 gnupg-l10n all 2.2.27-3ubuntu2.5
24.01 Ign:36 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 gnupg-utils amd64 2.2.27-3ubuntu2.5
24.01 Ign:37 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 gpg amd64 2.2.27-3ubuntu2.5
24.01 Ign:38 http://archive.ubuntu.com/ubuntu jammy/main amd64 pinentry-curses amd64 1.1.1-1build2
24.01 Ign:39 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 gpg-agent amd64 2.2.27-3ubuntu2.5
24.01 Ign:40 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 gpg-wks-client amd64 2.2.27-3ubuntu2.5
24.01 Ign:41 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 gpg-wks-server amd64 2.2.27-3ubuntu2.5
24.01 Ign:42 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 gpgsm amd64 2.2.27-3ubuntu2.5
24.01 Ign:43 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 gnupg all 2.2.27-3ubuntu2.5
24.01 Ign:44 http://archive.ubuntu.com/ubuntu jammy/main amd64 libstemmer0d amd64 2.2.0-1build1
24.01 Ign:45 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libxmlb2 amd64 0.3.24-1~ubuntu0.22.04.1
24.01 Ign:46 http://archive.ubuntu.com/ubuntu jammy/main amd64 libappstream4 amd64 0.15.2-2
24.01 Ign:47 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libdw1 amd64 0.186-1ubuntu0.1
24.01 Ign:48 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libglib2.0-bin amd64 2.72.4-0ubuntu2.9
24.01 Ign:49 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libunwind8 amd64 1.3.2-2build2.1
24.01 Ign:50 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libgstreamer1.0-0 amd64 1.20.3-0ubuntu1.1
24.01 Ign:51 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libpolkit-gobject-1-0 amd64 0.105-33ubuntu0.1
24.01 Ign:52 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libpolkit-agent-1-0 amd64 0.105-33ubuntu0.1
24.01 Ign:53 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 polkitd amd64 0.105-33ubuntu0.1
24.01 Ign:54 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 pkexec amd64 0.105-33ubuntu0.1
24.01 Ign:55 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 policykit-1 amd64 0.105-33ubuntu0.1
24.01 Ign:56 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 packagekit amd64 1.2.5-2ubuntu3.1
24.01 Ign:57 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 packagekit-tools amd64 1.2.5-2ubuntu3.1
24.01 Ign:58 http://archive.ubuntu.com/ubuntu jammy/main amd64 python3-blinker all 1.4+dfsg1-0.4
24.01 Ign:59 http://archive.ubuntu.com/ubuntu jammy/main amd64 python3-cffi-backend amd64 1.15.0-1build2
24.01 Ign:60 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 python3-cryptography amd64 3.4.8-1ubuntu2.4
24.01 Ign:61 http://archive.ubuntu.com/ubuntu jammy/main amd64 python3-pyparsing all 2.4.7-1
24.01 Ign:62 http://archive.ubuntu.com/ubuntu jammy/main amd64 python3-httplib2 all 0.20.2-2
24.01 Ign:63 http://archive.ubuntu.com/ubuntu jammy/main amd64 python3-more-itertools all 8.10.0-2
24.01 Ign:64 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 python3-zipp all 1.0.0-3ubuntu0.1
24.01 Ign:65 http://archive.ubuntu.com/ubuntu jammy/main amd64 python3-importlib-metadata all 4.6.4-1
24.01 Ign:66 http://archive.ubuntu.com/ubuntu jammy/main amd64 python3-jeepney all 0.7.1-3
24.01 Ign:67 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 python3-jwt all 2.3.0-1ubuntu0.3
24.01 Ign:68 http://archive.ubuntu.com/ubuntu jammy/main amd64 python3-secretstorage all 3.3.1-1
24.01 Ign:69 http://archive.ubuntu.com/ubuntu jammy/main amd64 python3-keyring all 23.5.0-1
24.01 Ign:70 http://archive.ubuntu.com/ubuntu jammy/main amd64 python3-lazr.uri all 1.0.6-2
24.01 Ign:71 http://archive.ubuntu.com/ubuntu jammy/main amd64 python3-wadllib all 1.3.6-1
24.01 Ign:72 http://archive.ubuntu.com/ubuntu jammy/main amd64 python3-distro all 1.7.0-1
24.01 Ign:73 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 python3-oauthlib all 3.2.0-1ubuntu0.1
24.01 Ign:74 http://archive.ubuntu.com/ubuntu jammy/main amd64 python3-six all 1.16.0-3ubuntu1
24.01 Ign:75 http://archive.ubuntu.com/ubuntu jammy/main amd64 python3-lazr.restfulclient all 0.14.4-1
24.01 Ign:76 http://archive.ubuntu.com/ubuntu jammy/main amd64 python3-launchpadlib all 1.10.16-1
24.01 Ign:77 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 python3-software-properties all 0.99.22.9
24.01 Ign:78 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 software-properties-common all 0.99.22.9
24.02 Ign:79 http://archive.ubuntu.com/ubuntu jammy/main amd64 unattended-upgrades all 2.8ubuntu1
25.00 Ign:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libsystemd0 amd64 249.11-0ubuntu3.21
25.00 Ign:2 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libapparmor1 amd64 3.0.4-2ubuntu2.5
25.00 Ign:3 http://archive.ubuntu.com/ubuntu jammy/main amd64 libargon2-1 amd64 0~20171227-0.3
25.00 Ign:4 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libdevmapper1.02.1 amd64 2:1.02.175-2.1ubuntu5
25.00 Ign:5 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libjson-c5 amd64 0.15-3~ubuntu1.22.04.2
25.00 Ign:6 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libcryptsetup12 amd64 2:2.4.3-1ubuntu1.3
25.00 Ign:7 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libip4tc2 amd64 1.8.7-1ubuntu5.2
25.00 Ign:8 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libkmod2 amd64 29-1ubuntu1.1
25.00 Ign:9 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 systemd amd64 249.11-0ubuntu3.21
25.01 Ign:10 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 systemd-sysv amd64 249.11-0ubuntu3.21
25.01 Ign:11 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libdbus-1-3 amd64 1.12.20-2ubuntu4.1
25.01 Ign:12 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 dbus amd64 1.12.20-2ubuntu4.1
25.01 Ign:13 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 distro-info-data all 0.52ubuntu0.12
25.01 Ign:14 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 dmsetup amd64 2:1.02.175-2.1ubuntu5
25.01 Ign:15 http://archive.ubuntu.com/ubuntu jammy/main amd64 libgirepository-1.0-1 amd64 1.72.0-1
25.01 Ign:16 http://archive.ubuntu.com/ubuntu jammy/main amd64 gir1.2-glib-2.0 amd64 1.72.0-1
25.01 Ign:17 http://archive.ubuntu.com/ubuntu jammy/main amd64 iso-codes all 4.9.0-1
25.01 Ign:18 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libnss-systemd amd64 249.11-0ubuntu3.21
25.01 Ign:19 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libpam-systemd amd64 249.11-0ubuntu3.21
25.01 Ign:20 http://archive.ubuntu.com/ubuntu jammy/main amd64 lsb-release all 11.1.0ubuntu4
25.01 Ign:21 http://archive.ubuntu.com/ubuntu jammy/main amd64 python3-dbus amd64 1.2.18-3build1
25.01 Ign:22 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 python3-gi amd64 3.42.1-0ubuntu1
25.01 Ign:23 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 networkd-dispatcher all 2.1-2ubuntu0.22.04.2
25.01 Ign:24 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 python-apt-common all 2.4.0ubuntu4.1
25.01 Ign:25 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 python3-apt amd64 2.4.0ubuntu4.1
25.01 Ign:26 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 systemd-timesyncd amd64 249.11-0ubuntu3.21
25.01 Ign:27 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 python3-distro-info all 1.1ubuntu0.2
25.01 Ign:28 http://archive.ubuntu.com/ubuntu jammy/main amd64 libassuan0 amd64 2.5.5-1build1
25.01 Ign:29 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 gpgconf amd64 2.2.27-3ubuntu2.5
25.01 Ign:30 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libksba8 amd64 1.6.0-2ubuntu0.2
25.01 Ign:31 http://archive.ubuntu.com/ubuntu jammy/main amd64 libnpth0 amd64 1.6-3build2
25.01 Ign:32 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 dirmngr amd64 2.2.27-3ubuntu2.5
25.01 Ign:33 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libpackagekit-glib2-18 amd64 1.2.5-2ubuntu3.1
25.01 Ign:34 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 gir1.2-packagekitglib-1.0 amd64 1.2.5-2ubuntu3.1
25.01 Ign:35 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 gnupg-l10n all 2.2.27-3ubuntu2.5
25.01 Ign:36 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 gnupg-utils amd64 2.2.27-3ubuntu2.5
25.01 Ign:37 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 gpg amd64 2.2.27-3ubuntu2.5
25.01 Ign:38 http://archive.ubuntu.com/ubuntu jammy/main amd64 pinentry-curses amd64 1.1.1-1build2
25.01 Ign:39 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 gpg-agent amd64 2.2.27-3ubuntu2.5
25.01 Ign:40 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 gpg-wks-client amd64 2.2.27-3ubuntu2.5
25.01 Ign:41 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 gpg-wks-server amd64 2.2.27-3ubuntu2.5
25.01 Ign:42 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 gpgsm amd64 2.2.27-3ubuntu2.5
25.01 Ign:43 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 gnupg all 2.2.27-3ubuntu2.5
25.01 Ign:44 http://archive.ubuntu.com/ubuntu jammy/main amd64 libstemmer0d amd64 2.2.0-1build1
25.01 Ign:45 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libxmlb2 amd64 0.3.24-1~ubuntu0.22.04.1
25.01 Ign:46 http://archive.ubuntu.com/ubuntu jammy/main amd64 libappstream4 amd64 0.15.2-2
25.01 Ign:47 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libdw1 amd64 0.186-1ubuntu0.1
25.01 Ign:48 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libglib2.0-bin amd64 2.72.4-0ubuntu2.9
25.01 Ign:49 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libunwind8 amd64 1.3.2-2build2.1
25.01 Ign:50 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libgstreamer1.0-0 amd64 1.20.3-0ubuntu1.1
25.01 Ign:51 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libpolkit-gobject-1-0 amd64 0.105-33ubuntu0.1
25.01 Ign:52 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libpolkit-agent-1-0 amd64 0.105-33ubuntu0.1
25.01 Ign:53 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 polkitd amd64 0.105-33ubuntu0.1
25.01 Ign:54 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 pkexec amd64 0.105-33ubuntu0.1
25.01 Ign:55 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 policykit-1 amd64 0.105-33ubuntu0.1
25.01 Ign:56 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 packagekit amd64 1.2.5-2ubuntu3.1
25.01 Ign:57 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 packagekit-tools amd64 1.2.5-2ubuntu3.1
25.01 Ign:58 http://archive.ubuntu.com/ubuntu jammy/main amd64 python3-blinker all 1.4+dfsg1-0.4
25.01 Ign:59 http://archive.ubuntu.com/ubuntu jammy/main amd64 python3-cffi-backend amd64 1.15.0-1build2
25.01 Ign:60 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 python3-cryptography amd64 3.4.8-1ubuntu2.4
25.01 Ign:61 http://archive.ubuntu.com/ubuntu jammy/main amd64 python3-pyparsing all 2.4.7-1
25.01 Ign:62 http://archive.ubuntu.com/ubuntu jammy/main amd64 python3-httplib2 all 0.20.2-2
25.01 Ign:63 http://archive.ubuntu.com/ubuntu jammy/main amd64 python3-more-itertools all 8.10.0-2
25.01 Ign:64 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 python3-zipp all 1.0.0-3ubuntu0.1
25.01 Ign:65 http://archive.ubuntu.com/ubuntu jammy/main amd64 python3-importlib-metadata all 4.6.4-1
25.01 Ign:66 http://archive.ubuntu.com/ubuntu jammy/main amd64 python3-jeepney all 0.7.1-3
25.01 Ign:67 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 python3-jwt all 2.3.0-1ubuntu0.3
25.01 Ign:68 http://archive.ubuntu.com/ubuntu jammy/main amd64 python3-secretstorage all 3.3.1-1
25.01 Ign:69 http://archive.ubuntu.com/ubuntu jammy/main amd64 python3-keyring all 23.5.0-1
25.01 Ign:70 http://archive.ubuntu.com/ubuntu jammy/main amd64 python3-lazr.uri all 1.0.6-2
25.01 Ign:71 http://archive.ubuntu.com/ubuntu jammy/main amd64 python3-wadllib all 1.3.6-1
25.01 Ign:72 http://archive.ubuntu.com/ubuntu jammy/main amd64 python3-distro all 1.7.0-1
25.01 Ign:73 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 python3-oauthlib all 3.2.0-1ubuntu0.1
25.01 Ign:74 http://archive.ubuntu.com/ubuntu jammy/main amd64 python3-six all 1.16.0-3ubuntu1
25.01 Ign:75 http://archive.ubuntu.com/ubuntu jammy/main amd64 python3-lazr.restfulclient all 0.14.4-1
25.02 Ign:76 http://archive.ubuntu.com/ubuntu jammy/main amd64 python3-launchpadlib all 1.10.16-1
25.02 Ign:77 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 python3-software-properties all 0.99.22.9
25.02 Ign:78 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 software-properties-common all 0.99.22.9
25.02 Ign:79 http://archive.ubuntu.com/ubuntu jammy/main amd64 unattended-upgrades all 2.8ubuntu1
27.00 Ign:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libsystemd0 amd64 249.11-0ubuntu3.21
27.00 Ign:2 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libapparmor1 amd64 3.0.4-2ubuntu2.5
27.00 Ign:3 http://archive.ubuntu.com/ubuntu jammy/main amd64 libargon2-1 amd64 0~20171227-0.3
27.00 Ign:4 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libdevmapper1.02.1 amd64 2:1.02.175-2.1ubuntu5
27.00 Ign:5 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libjson-c5 amd64 0.15-3~ubuntu1.22.04.2
27.00 Ign:6 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libcryptsetup12 amd64 2:2.4.3-1ubuntu1.3
27.00 Ign:7 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libip4tc2 amd64 1.8.7-1ubuntu5.2
27.00 Ign:8 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libkmod2 amd64 29-1ubuntu1.1
27.00 Ign:9 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 systemd amd64 249.11-0ubuntu3.21
27.00 Ign:10 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 systemd-sysv amd64 249.11-0ubuntu3.21
27.00 Ign:11 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libdbus-1-3 amd64 1.12.20-2ubuntu4.1
27.00 Ign:12 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 dbus amd64 1.12.20-2ubuntu4.1
27.00 Ign:13 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 distro-info-data all 0.52ubuntu0.12
27.00 Ign:14 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 dmsetup amd64 2:1.02.175-2.1ubuntu5
27.00 Ign:15 http://archive.ubuntu.com/ubuntu jammy/main amd64 libgirepository-1.0-1 amd64 1.72.0-1
27.00 Ign:16 http://archive.ubuntu.com/ubuntu jammy/main amd64 gir1.2-glib-2.0 amd64 1.72.0-1
27.00 Ign:17 http://archive.ubuntu.com/ubuntu jammy/main amd64 iso-codes all 4.9.0-1
27.01 Ign:18 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libnss-systemd amd64 249.11-0ubuntu3.21
27.01 Ign:19 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libpam-systemd amd64 249.11-0ubuntu3.21
27.01 Ign:20 http://archive.ubuntu.com/ubuntu jammy/main amd64 lsb-release all 11.1.0ubuntu4
27.01 Ign:21 http://archive.ubuntu.com/ubuntu jammy/main amd64 python3-dbus amd64 1.2.18-3build1
27.01 Ign:22 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 python3-gi amd64 3.42.1-0ubuntu1
27.01 Ign:23 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 networkd-dispatcher all 2.1-2ubuntu0.22.04.2
27.01 Ign:24 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 python-apt-common all 2.4.0ubuntu4.1
27.01 Ign:25 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 python3-apt amd64 2.4.0ubuntu4.1
27.01 Ign:26 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 systemd-timesyncd amd64 249.11-0ubuntu3.21
27.01 Ign:27 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 python3-distro-info all 1.1ubuntu0.2
27.01 Ign:28 http://archive.ubuntu.com/ubuntu jammy/main amd64 libassuan0 amd64 2.5.5-1build1
27.01 Ign:29 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 gpgconf amd64 2.2.27-3ubuntu2.5
27.01 Ign:30 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libksba8 amd64 1.6.0-2ubuntu0.2
27.01 Ign:31 http://archive.ubuntu.com/ubuntu jammy/main amd64 libnpth0 amd64 1.6-3build2
27.01 Ign:32 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 dirmngr amd64 2.2.27-3ubuntu2.5
27.01 Ign:33 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libpackagekit-glib2-18 amd64 1.2.5-2ubuntu3.1
27.01 Ign:34 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 gir1.2-packagekitglib-1.0 amd64 1.2.5-2ubuntu3.1
27.01 Ign:35 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 gnupg-l10n all 2.2.27-3ubuntu2.5
27.01 Ign:36 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 gnupg-utils amd64 2.2.27-3ubuntu2.5
27.01 Ign:37 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 gpg amd64 2.2.27-3ubuntu2.5
27.01 Ign:38 http://archive.ubuntu.com/ubuntu jammy/main amd64 pinentry-curses amd64 1.1.1-1build2
27.01 Ign:39 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 gpg-agent amd64 2.2.27-3ubuntu2.5
27.01 Ign:40 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 gpg-wks-client amd64 2.2.27-3ubuntu2.5
27.01 Ign:41 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 gpg-wks-server amd64 2.2.27-3ubuntu2.5
27.01 Ign:42 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 gpgsm amd64 2.2.27-3ubuntu2.5
27.01 Ign:43 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 gnupg all 2.2.27-3ubuntu2.5
27.01 Ign:44 http://archive.ubuntu.com/ubuntu jammy/main amd64 libstemmer0d amd64 2.2.0-1build1
27.01 Ign:45 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libxmlb2 amd64 0.3.24-1~ubuntu0.22.04.1
27.01 Ign:46 http://archive.ubuntu.com/ubuntu jammy/main amd64 libappstream4 amd64 0.15.2-2
27.01 Ign:47 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libdw1 amd64 0.186-1ubuntu0.1
27.01 Ign:48 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libglib2.0-bin amd64 2.72.4-0ubuntu2.9
27.01 Ign:49 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libunwind8 amd64 1.3.2-2build2.1
27.01 Ign:50 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libgstreamer1.0-0 amd64 1.20.3-0ubuntu1.1
27.01 Ign:51 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libpolkit-gobject-1-0 amd64 0.105-33ubuntu0.1
27.01 Ign:52 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libpolkit-agent-1-0 amd64 0.105-33ubuntu0.1
27.01 Ign:53 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 polkitd amd64 0.105-33ubuntu0.1
27.01 Ign:54 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 pkexec amd64 0.105-33ubuntu0.1
27.01 Ign:55 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 policykit-1 amd64 0.105-33ubuntu0.1
27.01 Ign:56 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 packagekit amd64 1.2.5-2ubuntu3.1
27.01 Ign:57 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 packagekit-tools amd64 1.2.5-2ubuntu3.1
27.01 Ign:58 http://archive.ubuntu.com/ubuntu jammy/main amd64 python3-blinker all 1.4+dfsg1-0.4
27.01 Ign:59 http://archive.ubuntu.com/ubuntu jammy/main amd64 python3-cffi-backend amd64 1.15.0-1build2
27.01 Ign:60 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 python3-cryptography amd64 3.4.8-1ubuntu2.4
27.01 Ign:61 http://archive.ubuntu.com/ubuntu jammy/main amd64 python3-pyparsing all 2.4.7-1
27.01 Ign:62 http://archive.ubuntu.com/ubuntu jammy/main amd64 python3-httplib2 all 0.20.2-2
27.01 Ign:63 http://archive.ubuntu.com/ubuntu jammy/main amd64 python3-more-itertools all 8.10.0-2
27.01 Ign:64 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 python3-zipp all 1.0.0-3ubuntu0.1
27.01 Ign:65 http://archive.ubuntu.com/ubuntu jammy/main amd64 python3-importlib-metadata all 4.6.4-1
27.01 Ign:66 http://archive.ubuntu.com/ubuntu jammy/main amd64 python3-jeepney all 0.7.1-3
27.01 Ign:67 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 python3-jwt all 2.3.0-1ubuntu0.3
27.01 Ign:68 http://archive.ubuntu.com/ubuntu jammy/main amd64 python3-secretstorage all 3.3.1-1
27.02 Ign:69 http://archive.ubuntu.com/ubuntu jammy/main amd64 python3-keyring all 23.5.0-1
27.02 Ign:70 http://archive.ubuntu.com/ubuntu jammy/main amd64 python3-lazr.uri all 1.0.6-2
27.02 Ign:71 http://archive.ubuntu.com/ubuntu jammy/main amd64 python3-wadllib all 1.3.6-1
27.02 Ign:72 http://archive.ubuntu.com/ubuntu jammy/main amd64 python3-distro all 1.7.0-1
27.02 Ign:73 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 python3-oauthlib all 3.2.0-1ubuntu0.1
27.02 Ign:74 http://archive.ubuntu.com/ubuntu jammy/main amd64 python3-six all 1.16.0-3ubuntu1
27.02 Ign:75 http://archive.ubuntu.com/ubuntu jammy/main amd64 python3-lazr.restfulclient all 0.14.4-1
27.02 Ign:76 http://archive.ubuntu.com/ubuntu jammy/main amd64 python3-launchpadlib all 1.10.16-1
27.02 Ign:77 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 python3-software-properties all 0.99.22.9
27.02 Ign:78 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 software-properties-common all 0.99.22.9
27.02 Ign:79 http://archive.ubuntu.com/ubuntu jammy/main amd64 unattended-upgrades all 2.8ubuntu1
31.01 Ign:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libsystemd0 amd64 249.11-0ubuntu3.21
31.02 Err:2 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libapparmor1 amd64 3.0.4-2ubuntu2.5
31.02   Could not resolve 'archive.ubuntu.com'
31.02 Err:3 http://archive.ubuntu.com/ubuntu jammy/main amd64 libargon2-1 amd64 0~20171227-0.3
31.02   Could not resolve 'archive.ubuntu.com'
31.02 Err:4 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libdevmapper1.02.1 amd64 2:1.02.175-2.1ubuntu5
31.02   Could not resolve 'archive.ubuntu.com'
31.03 Ign:5 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libjson-c5 amd64 0.15-3~ubuntu1.22.04.2
31.03 Err:6 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libcryptsetup12 amd64 2:2.4.3-1ubuntu1.3
31.03   Could not resolve 'archive.ubuntu.com'
31.03 Err:7 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libip4tc2 amd64 1.8.7-1ubuntu5.2
31.03   Could not resolve 'archive.ubuntu.com'
31.03 Ign:8 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libkmod2 amd64 29-1ubuntu1.1
31.03 Ign:9 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 systemd amd64 249.11-0ubuntu3.21
31.03 Ign:10 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 systemd-sysv amd64 249.11-0ubuntu3.21
31.03 Ign:11 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libdbus-1-3 amd64 1.12.20-2ubuntu4.1
31.03 Ign:12 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 dbus amd64 1.12.20-2ubuntu4.1
31.03 Ign:13 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 distro-info-data all 0.52ubuntu0.12
31.03 Err:14 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 dmsetup amd64 2:1.02.175-2.1ubuntu5
31.03   Could not resolve 'archive.ubuntu.com'
31.03 Err:15 http://archive.ubuntu.com/ubuntu jammy/main amd64 libgirepository-1.0-1 amd64 1.72.0-1
31.03   Could not resolve 'archive.ubuntu.com'
31.03 Err:16 http://archive.ubuntu.com/ubuntu jammy/main amd64 gir1.2-glib-2.0 amd64 1.72.0-1
31.03   Could not resolve 'archive.ubuntu.com'
31.03 Err:17 http://archive.ubuntu.com/ubuntu jammy/main amd64 iso-codes all 4.9.0-1
31.03   Could not resolve 'archive.ubuntu.com'
31.03 Ign:18 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libnss-systemd amd64 249.11-0ubuntu3.21
31.03 Ign:19 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libpam-systemd amd64 249.11-0ubuntu3.21
31.03 Err:20 http://archive.ubuntu.com/ubuntu jammy/main amd64 lsb-release all 11.1.0ubuntu4
31.03   Could not resolve 'archive.ubuntu.com'
31.03 Err:21 http://archive.ubuntu.com/ubuntu jammy/main amd64 python3-dbus amd64 1.2.18-3build1
31.03   Could not resolve 'archive.ubuntu.com'
31.03 Err:22 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 python3-gi amd64 3.42.1-0ubuntu1
31.03   Could not resolve 'archive.ubuntu.com'
31.03 Ign:23 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 networkd-dispatcher all 2.1-2ubuntu0.22.04.2
31.03 Ign:24 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 python-apt-common all 2.4.0ubuntu4.1
31.03 Ign:25 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 python3-apt amd64 2.4.0ubuntu4.1
31.03 Ign:26 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 systemd-timesyncd amd64 249.11-0ubuntu3.21
31.03 Err:27 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 python3-distro-info all 1.1ubuntu0.2
31.03   Could not resolve 'archive.ubuntu.com'
31.03 Err:28 http://archive.ubuntu.com/ubuntu jammy/main amd64 libassuan0 amd64 2.5.5-1build1
31.03   Could not resolve 'archive.ubuntu.com'
31.03 Ign:29 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 gpgconf amd64 2.2.27-3ubuntu2.5
31.03 Ign:30 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libksba8 amd64 1.6.0-2ubuntu0.2
31.03 Err:31 http://archive.ubuntu.com/ubuntu jammy/main amd64 libnpth0 amd64 1.6-3build2
31.03   Could not resolve 'archive.ubuntu.com'
31.03 Ign:32 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 dirmngr amd64 2.2.27-3ubuntu2.5
31.03 Ign:33 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libpackagekit-glib2-18 amd64 1.2.5-2ubuntu3.1
31.03 Ign:34 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 gir1.2-packagekitglib-1.0 amd64 1.2.5-2ubuntu3.1
31.03 Ign:35 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 gnupg-l10n all 2.2.27-3ubuntu2.5
31.03 Ign:36 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 gnupg-utils amd64 2.2.27-3ubuntu2.5
31.03 Ign:37 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 gpg amd64 2.2.27-3ubuntu2.5
31.03 Err:38 http://archive.ubuntu.com/ubuntu jammy/main amd64 pinentry-curses amd64 1.1.1-1build2
31.03   Could not resolve 'archive.ubuntu.com'
31.03 Ign:39 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 gpg-agent amd64 2.2.27-3ubuntu2.5
31.03 Ign:40 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 gpg-wks-client amd64 2.2.27-3ubuntu2.5
31.03 Ign:41 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 gpg-wks-server amd64 2.2.27-3ubuntu2.5
31.03 Ign:42 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 gpgsm amd64 2.2.27-3ubuntu2.5
31.03 Ign:43 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 gnupg all 2.2.27-3ubuntu2.5
31.03 Err:44 http://archive.ubuntu.com/ubuntu jammy/main amd64 libstemmer0d amd64 2.2.0-1build1
31.03   Could not resolve 'archive.ubuntu.com'
31.03 Err:45 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libxmlb2 amd64 0.3.24-1~ubuntu0.22.04.1
31.03   Could not resolve 'archive.ubuntu.com'
31.03 Err:46 http://archive.ubuntu.com/ubuntu jammy/main amd64 libappstream4 amd64 0.15.2-2
31.03   Could not resolve 'archive.ubuntu.com'
31.03 Ign:47 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libdw1 amd64 0.186-1ubuntu0.1
31.03 Ign:48 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libglib2.0-bin amd64 2.72.4-0ubuntu2.9
31.03 Err:49 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libunwind8 amd64 1.3.2-2build2.1
31.03   Could not resolve 'archive.ubuntu.com'
31.03 Ign:50 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libgstreamer1.0-0 amd64 1.20.3-0ubuntu1.1
31.03 Ign:51 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libpolkit-gobject-1-0 amd64 0.105-33ubuntu0.1
31.03 Ign:52 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libpolkit-agent-1-0 amd64 0.105-33ubuntu0.1
31.03 Ign:53 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 polkitd amd64 0.105-33ubuntu0.1
31.03 Ign:54 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 pkexec amd64 0.105-33ubuntu0.1
31.03 Ign:55 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 policykit-1 amd64 0.105-33ubuntu0.1
31.03 Ign:56 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 packagekit amd64 1.2.5-2ubuntu3.1
31.03 Ign:57 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 packagekit-tools amd64 1.2.5-2ubuntu3.1
31.03 Err:58 http://archive.ubuntu.com/ubuntu jammy/main amd64 python3-blinker all 1.4+dfsg1-0.4
31.03   Could not resolve 'archive.ubuntu.com'
31.03 Err:59 http://archive.ubuntu.com/ubuntu jammy/main amd64 python3-cffi-backend amd64 1.15.0-1build2
31.03   Could not resolve 'archive.ubuntu.com'
31.03 Ign:60 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 python3-cryptography amd64 3.4.8-1ubuntu2.4
31.03 Err:61 http://archive.ubuntu.com/ubuntu jammy/main amd64 python3-pyparsing all 2.4.7-1
31.03   Could not resolve 'archive.ubuntu.com'
31.03 Err:62 http://archive.ubuntu.com/ubuntu jammy/main amd64 python3-httplib2 all 0.20.2-2
31.03   Could not resolve 'archive.ubuntu.com'
31.03 Err:63 http://archive.ubuntu.com/ubuntu jammy/main amd64 python3-more-itertools all 8.10.0-2
31.03   Could not resolve 'archive.ubuntu.com'
31.03 Ign:64 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 python3-zipp all 1.0.0-3ubuntu0.1
31.03 Err:65 http://archive.ubuntu.com/ubuntu jammy/main amd64 python3-importlib-metadata all 4.6.4-1
31.03   Could not resolve 'archive.ubuntu.com'
31.03 Err:66 http://archive.ubuntu.com/ubuntu jammy/main amd64 python3-jeepney all 0.7.1-3
31.03   Could not resolve 'archive.ubuntu.com'
31.03 Ign:67 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 python3-jwt all 2.3.0-1ubuntu0.3
31.03 Err:68 http://archive.ubuntu.com/ubuntu jammy/main amd64 python3-secretstorage all 3.3.1-1
31.03   Could not resolve 'archive.ubuntu.com'
31.03 Err:69 http://archive.ubuntu.com/ubuntu jammy/main amd64 python3-keyring all 23.5.0-1
31.03   Could not resolve 'archive.ubuntu.com'
31.03 Err:70 http://archive.ubuntu.com/ubuntu jammy/main amd64 python3-lazr.uri all 1.0.6-2
31.03   Could not resolve 'archive.ubuntu.com'
31.03 Err:71 http://archive.ubuntu.com/ubuntu jammy/main amd64 python3-wadllib all 1.3.6-1
31.03   Could not resolve 'archive.ubuntu.com'
31.03 Err:72 http://archive.ubuntu.com/ubuntu jammy/main amd64 python3-distro all 1.7.0-1
31.03   Could not resolve 'archive.ubuntu.com'
31.03 Ign:73 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 python3-oauthlib all 3.2.0-1ubuntu0.1
31.03 Err:74 http://archive.ubuntu.com/ubuntu jammy/main amd64 python3-six all 1.16.0-3ubuntu1
31.03   Could not resolve 'archive.ubuntu.com'
31.03 Err:75 http://archive.ubuntu.com/ubuntu jammy/main amd64 python3-lazr.restfulclient all 0.14.4-1
31.03   Could not resolve 'archive.ubuntu.com'
31.03 Err:76 http://archive.ubuntu.com/ubuntu jammy/main amd64 python3-launchpadlib all 1.10.16-1
31.03   Could not resolve 'archive.ubuntu.com'
31.03 Err:77 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 python3-software-properties all 0.99.22.9
31.03   Could not resolve 'archive.ubuntu.com'
31.03 Err:78 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 software-properties-common all 0.99.22.9
31.03   Could not resolve 'archive.ubuntu.com'
31.03 Err:79 http://archive.ubuntu.com/ubuntu jammy/main amd64 unattended-upgrades all 2.8ubuntu1
31.03   Could not resolve 'archive.ubuntu.com'
31.50 Get:1 http://security.ubuntu.com/ubuntu jammy-updates/main amd64 libsystemd0 amd64 249.11-0ubuntu3.21 [316 kB]
31.76 Get:5 http://security.ubuntu.com/ubuntu jammy-updates/main amd64 libjson-c5 amd64 0.15-3~ubuntu1.22.04.2 [33.5 kB]
31.78 Get:8 http://security.ubuntu.com/ubuntu jammy-updates/main amd64 libkmod2 amd64 29-1ubuntu1.1 [48.1 kB]
31.79 Get:9 http://security.ubuntu.com/ubuntu jammy-updates/main amd64 systemd amd64 249.11-0ubuntu3.21 [4585 kB]
32.68 Get:10 http://security.ubuntu.com/ubuntu jammy-updates/main amd64 systemd-sysv amd64 249.11-0ubuntu3.21 [10.5 kB]
32.68 Get:11 http://security.ubuntu.com/ubuntu jammy-updates/main amd64 libdbus-1-3 amd64 1.12.20-2ubuntu4.1 [189 kB]
32.81 Get:12 http://security.ubuntu.com/ubuntu jammy-updates/main amd64 dbus amd64 1.12.20-2ubuntu4.1 [158 kB]
32.86 Get:13 http://security.ubuntu.com/ubuntu jammy-updates/main amd64 distro-info-data all 0.52ubuntu0.12 [5488 B]
32.87 Get:18 http://security.ubuntu.com/ubuntu jammy-updates/main amd64 libnss-systemd amd64 249.11-0ubuntu3.21 [133 kB]
32.89 Get:19 http://security.ubuntu.com/ubuntu jammy-updates/main amd64 libpam-systemd amd64 249.11-0ubuntu3.21 [203 kB]
33.09 Get:23 http://security.ubuntu.com/ubuntu jammy-updates/main amd64 networkd-dispatcher all 2.1-2ubuntu0.22.04.2 [15.8 kB]
33.10 Get:24 http://security.ubuntu.com/ubuntu jammy-updates/main amd64 python-apt-common all 2.4.0ubuntu4.1 [14.7 kB]
33.11 Get:25 http://security.ubuntu.com/ubuntu jammy-updates/main amd64 python3-apt amd64 2.4.0ubuntu4.1 [164 kB]
33.15 Get:26 http://security.ubuntu.com/ubuntu jammy-updates/main amd64 systemd-timesyncd amd64 249.11-0ubuntu3.21 [31.2 kB]
33.16 Get:29 http://security.ubuntu.com/ubuntu jammy-updates/main amd64 gpgconf amd64 2.2.27-3ubuntu2.5 [94.3 kB]
33.16 Get:30 http://security.ubuntu.com/ubuntu jammy-updates/main amd64 libksba8 amd64 1.6.0-2ubuntu0.2 [119 kB]
33.17 Get:32 http://security.ubuntu.com/ubuntu jammy-updates/main amd64 dirmngr amd64 2.2.27-3ubuntu2.5 [293 kB]
33.20 Get:33 http://security.ubuntu.com/ubuntu jammy-updates/main amd64 libpackagekit-glib2-18 amd64 1.2.5-2ubuntu3.1 [124 kB]
33.21 Get:34 http://security.ubuntu.com/ubuntu jammy-updates/main amd64 gir1.2-packagekitglib-1.0 amd64 1.2.5-2ubuntu3.1 [25.3 kB]
33.21 Get:35 http://security.ubuntu.com/ubuntu jammy-updates/main amd64 gnupg-l10n all 2.2.27-3ubuntu2.5 [54.5 kB]
33.21 Get:36 http://security.ubuntu.com/ubuntu jammy-updates/main amd64 gnupg-utils amd64 2.2.27-3ubuntu2.5 [309 kB]
33.25 Get:37 http://security.ubuntu.com/ubuntu jammy-updates/main amd64 gpg amd64 2.2.27-3ubuntu2.5 [519 kB]
33.36 Get:39 http://security.ubuntu.com/ubuntu jammy-updates/main amd64 gpg-agent amd64 2.2.27-3ubuntu2.5 [209 kB]
33.40 Get:40 http://security.ubuntu.com/ubuntu jammy-updates/main amd64 gpg-wks-client amd64 2.2.27-3ubuntu2.5 [62.7 kB]
33.40 Get:41 http://security.ubuntu.com/ubuntu jammy-updates/main amd64 gpg-wks-server amd64 2.2.27-3ubuntu2.5 [57.6 kB]
33.41 Get:42 http://security.ubuntu.com/ubuntu jammy-updates/main amd64 gpgsm amd64 2.2.27-3ubuntu2.5 [197 kB]
33.43 Get:43 http://security.ubuntu.com/ubuntu jammy-updates/main amd64 gnupg all 2.2.27-3ubuntu2.5 [315 kB]
33.45 Get:47 http://security.ubuntu.com/ubuntu jammy-updates/main amd64 libdw1 amd64 0.186-1ubuntu0.1 [251 kB]
33.47 Get:48 http://security.ubuntu.com/ubuntu jammy-updates/main amd64 libglib2.0-bin amd64 2.72.4-0ubuntu2.9 [80.9 kB]
33.48 Get:50 http://security.ubuntu.com/ubuntu jammy-updates/main amd64 libgstreamer1.0-0 amd64 1.20.3-0ubuntu1.1 [984 kB]
33.62 Get:51 http://security.ubuntu.com/ubuntu jammy-updates/main amd64 libpolkit-gobject-1-0 amd64 0.105-33ubuntu0.1 [43.3 kB]
33.62 Get:52 http://security.ubuntu.com/ubuntu jammy-updates/main amd64 libpolkit-agent-1-0 amd64 0.105-33ubuntu0.1 [16.9 kB]
33.62 Get:53 http://security.ubuntu.com/ubuntu jammy-updates/main amd64 polkitd amd64 0.105-33ubuntu0.1 [80.1 kB]
33.62 Get:54 http://security.ubuntu.com/ubuntu jammy-updates/main amd64 pkexec amd64 0.105-33ubuntu0.1 [15.2 kB]
33.62 Get:55 http://security.ubuntu.com/ubuntu jammy-updates/main amd64 policykit-1 amd64 0.105-33ubuntu0.1 [2438 B]
33.62 Get:56 http://security.ubuntu.com/ubuntu jammy-updates/main amd64 packagekit amd64 1.2.5-2ubuntu3.1 [442 kB]
33.70 Get:57 http://security.ubuntu.com/ubuntu jammy-updates/main amd64 packagekit-tools amd64 1.2.5-2ubuntu3.1 [28.8 kB]
33.72 Get:60 http://security.ubuntu.com/ubuntu jammy-updates/main amd64 python3-cryptography amd64 3.4.8-1ubuntu2.4 [236 kB]
33.80 Get:64 http://security.ubuntu.com/ubuntu jammy-updates/main amd64 python3-zipp all 1.0.0-3ubuntu0.1 [6288 B]
33.81 Get:67 http://security.ubuntu.com/ubuntu jammy-updates/main amd64 python3-jwt all 2.3.0-1ubuntu0.3 [17.3 kB]
33.86 Get:73 http://security.ubuntu.com/ubuntu jammy-updates/main amd64 python3-oauthlib all 3.2.0-1ubuntu0.1 [89.9 kB]
33.90 Fetched 10.6 MB in 30s (353 kB/s)
33.90 E: Failed to fetch http://archive.ubuntu.com/ubuntu/pool/main/a/apparmor/libapparmor1_3.0.4-2ubuntu2.5_amd64.deb  Could not resolve 'archive.ubuntu.com'
33.90 E: Failed to fetch http://archive.ubuntu.com/ubuntu/pool/main/a/argon2/libargon2-1_0%7e20171227-0.3_amd64.deb  Could not resolve 'archive.ubuntu.com'
33.90 E: Failed to fetch http://archive.ubuntu.com/ubuntu/pool/main/l/lvm2/libdevmapper1.02.1_1.02.175-2.1ubuntu5_amd64.deb  Could not resolve 'archive.ubuntu.com'
33.90 E: Failed to fetch http://archive.ubuntu.com/ubuntu/pool/main/c/cryptsetup/libcryptsetup12_2.4.3-1ubuntu1.3_amd64.deb  Could not resolve 'archive.ubuntu.com'
33.90 E: Failed to fetch http://archive.ubuntu.com/ubuntu/pool/main/i/iptables/libip4tc2_1.8.7-1ubuntu5.2_amd64.deb  Could not resolve 'archive.ubuntu.com'
33.90 E: Failed to fetch http://archive.ubuntu.com/ubuntu/pool/main/l/lvm2/dmsetup_1.02.175-2.1ubuntu5_amd64.deb  Could not resolve 'archive.ubuntu.com'
33.90 E: Failed to fetch http://archive.ubuntu.com/ubuntu/pool/main/g/gobject-introspection/libgirepository-1.0-1_1.72.0-1_amd64.deb  Could not resolve 'archive.ubuntu.com'
33.90 E: Failed to fetch http://archive.ubuntu.com/ubuntu/pool/main/g/gobject-introspection/gir1.2-glib-2.0_1.72.0-1_amd64.deb  Could not resolve 'archive.ubuntu.com'
33.90 E: Failed to fetch http://archive.ubuntu.com/ubuntu/pool/main/i/iso-codes/iso-codes_4.9.0-1_all.deb  Could not resolve 'archive.ubuntu.com'
33.90 E: Failed to fetch http://archive.ubuntu.com/ubuntu/pool/main/l/lsb/lsb-release_11.1.0ubuntu4_all.deb  Could not resolve 'archive.ubuntu.com'
33.90 E: Failed to fetch http://archive.ubuntu.com/ubuntu/pool/main/d/dbus-python/python3-dbus_1.2.18-3build1_amd64.deb  Could not resolve 'archive.ubuntu.com'
33.90 E: Failed to fetch http://archive.ubuntu.com/ubuntu/pool/main/p/pygobject/python3-gi_3.42.1-0ubuntu1_amd64.deb  Could not resolve 'archive.ubuntu.com'
33.90 E: Failed to fetch http://archive.ubuntu.com/ubuntu/pool/main/d/distro-info/python3-distro-info_1.1ubuntu0.2_all.deb  Could not resolve 'archive.ubuntu.com'
33.90 E: Failed to fetch http://archive.ubuntu.com/ubuntu/pool/main/liba/libassuan/libassuan0_2.5.5-1build1_amd64.deb  Could not resolve 'archive.ubuntu.com'
33.90 E: Failed to fetch http://archive.ubuntu.com/ubuntu/pool/main/n/npth/libnpth0_1.6-3build2_amd64.deb  Could not resolve 'archive.ubuntu.com'
33.90 E: Failed to fetch http://archive.ubuntu.com/ubuntu/pool/main/p/pinentry/pinentry-curses_1.1.1-1build2_amd64.deb  Could not resolve 'archive.ubuntu.com'
33.90 E: Failed to fetch http://archive.ubuntu.com/ubuntu/pool/main/s/snowball/libstemmer0d_2.2.0-1build1_amd64.deb  Could not resolve 'archive.ubuntu.com'
33.90 E: Failed to fetch http://archive.ubuntu.com/ubuntu/pool/main/libx/libxmlb/libxmlb2_0.3.24-1%7eubuntu0.22.04.1_amd64.deb  Could not resolve 'archive.ubuntu.com'
33.90 E: Failed to fetch http://archive.ubuntu.com/ubuntu/pool/main/a/appstream/libappstream4_0.15.2-2_amd64.deb  Could not resolve 'archive.ubuntu.com'
33.90 E: Failed to fetch http://archive.ubuntu.com/ubuntu/pool/main/libu/libunwind/libunwind8_1.3.2-2build2.1_amd64.deb  Could not resolve 'archive.ubuntu.com'
33.90 E: Failed to fetch http://archive.ubuntu.com/ubuntu/pool/main/b/blinker/python3-blinker_1.4%2bdfsg1-0.4_all.deb  Could not resolve 'archive.ubuntu.com'
33.90 E: Failed to fetch http://archive.ubuntu.com/ubuntu/pool/main/p/python-cffi/python3-cffi-backend_1.15.0-1build2_amd64.deb  Could not resolve 'archive.ubuntu.com'
33.90 E: Failed to fetch http://archive.ubuntu.com/ubuntu/pool/main/p/pyparsing/python3-pyparsing_2.4.7-1_all.deb  Could not resolve 'archive.ubuntu.com'
33.90 E: Failed to fetch http://archive.ubuntu.com/ubuntu/pool/main/p/python-httplib2/python3-httplib2_0.20.2-2_all.deb  Could not resolve 'archive.ubuntu.com'
33.90 E: Failed to fetch http://archive.ubuntu.com/ubuntu/pool/main/m/more-itertools/python3-more-itertools_8.10.0-2_all.deb  Could not resolve 'archive.ubuntu.com'
33.90 E: Failed to fetch http://archive.ubuntu.com/ubuntu/pool/main/p/python-importlib-metadata/python3-importlib-metadata_4.6.4-1_all.deb  Could not resolve 'archive.ubuntu.com'
33.90 E: Failed to fetch http://archive.ubuntu.com/ubuntu/pool/main/j/jeepney/python3-jeepney_0.7.1-3_all.deb  Could not resolve 'archive.ubuntu.com'
33.90 E: Failed to fetch http://archive.ubuntu.com/ubuntu/pool/main/p/python-secretstorage/python3-secretstorage_3.3.1-1_all.deb  Could not resolve 'archive.ubuntu.com'
33.90 E: Failed to fetch http://archive.ubuntu.com/ubuntu/pool/main/p/python-keyring/python3-keyring_23.5.0-1_all.deb  Could not resolve 'archive.ubuntu.com'
33.90 E: Failed to fetch http://archive.ubuntu.com/ubuntu/pool/main/l/lazr.uri/python3-lazr.uri_1.0.6-2_all.deb  Could not resolve 'archive.ubuntu.com'
33.90 E: Failed to fetch http://archive.ubuntu.com/ubuntu/pool/main/p/python-wadllib/python3-wadllib_1.3.6-1_all.deb  Could not resolve 'archive.ubuntu.com'
33.90 E: Failed to fetch http://archive.ubuntu.com/ubuntu/pool/main/p/python-distro/python3-distro_1.7.0-1_all.deb  Could not resolve 'archive.ubuntu.com'
33.90 E: Failed to fetch http://archive.ubuntu.com/ubuntu/pool/main/s/six/python3-six_1.16.0-3ubuntu1_all.deb  Could not resolve 'archive.ubuntu.com'
33.90 E: Failed to fetch http://archive.ubuntu.com/ubuntu/pool/main/l/lazr.restfulclient/python3-lazr.restfulclient_0.14.4-1_all.deb  Could not resolve 'archive.ubuntu.com'
33.90 E: Failed to fetch http://archive.ubuntu.com/ubuntu/pool/main/p/python-launchpadlib/python3-launchpadlib_1.10.16-1_all.deb  Could not resolve 'archive.ubuntu.com'
33.90 E: Failed to fetch http://archive.ubuntu.com/ubuntu/pool/main/s/software-properties/python3-software-properties_0.99.22.9_all.deb  Could not resolve 'archive.ubuntu.com'
33.90 E: Failed to fetch http://archive.ubuntu.com/ubuntu/pool/main/s/software-properties/software-properties-common_0.99.22.9_all.deb  Could not resolve 'archive.ubuntu.com'
33.90 E: Failed to fetch http://archive.ubuntu.com/ubuntu/pool/main/u/unattended-upgrades/unattended-upgrades_2.8ubuntu1_all.deb  Could not resolve 'archive.ubuntu.com'
33.90 E: Unable to fetch some archives, maybe run apt-get update or try with --fix-missing?
------
Dockerfile:16
--------------------
  15 |     ENV PATH="$PYENV_ROOT/bin:$PATH"
  16 | >>> RUN apt install -y software-properties-common \
  17 | >>>     && apt update \
  18 | >>>     && add-apt-repository -y ppa:deadsnakes/ppa \
  19 | >>>     && apt update \
  20 | >>>     && apt install -y python3.10 python3.10-distutils \
  21 | >>>     # Remove python3-blinker to avoid pip uninstall conflicts
  22 | >>>     && apt remove -y python3-blinker \
  23 | >>>     && ln -s -f $(which python3.10) /usr/bin/python \
  24 | >>>     && wget https://bootstrap.pypa.io/get-pip.py -O /tmp/get-pip.py \
  25 | >>>     && python /tmp/get-pip.py
  26 |     
--------------------
ERROR: failed to build: failed to solve: process "/bin/sh -c apt install -y software-properties-common     && apt update     && add-apt-repository -y ppa:deadsnakes/ppa     && apt update     && apt install -y python3.10 python3.10-distutils     && apt remove -y python3-blinker     && ln -s -f $(which python3.10) /usr/bin/python     && wget https://bootstrap.pypa.io/get-pip.py -O /tmp/get-pip.py     && python /tmp/get-pip.py" did not complete successfully: exit code: 100
Traceback (most recent call last):
  File "/home/zelitosaide/Documents/mlops/.venv/bin/mlflow", line 8, in <module>
    sys.exit(cli())
             ~~~^^
  File "/home/zelitosaide/Documents/mlops/.venv/lib/python3.14/site-packages/click/core.py", line 1524, in __call__
    return self.main(*args, **kwargs)
           ~~~~~~~~~^^^^^^^^^^^^^^^^^
  File "/home/zelitosaide/Documents/mlops/.venv/lib/python3.14/site-packages/click/core.py", line 1445, in main
    rv = self.invoke(ctx)
  File "/home/zelitosaide/Documents/mlops/.venv/lib/python3.14/site-packages/click/core.py", line 1912, in invoke
    return _process_result(sub_ctx.command.invoke(sub_ctx))
                           ~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^
  File "/home/zelitosaide/Documents/mlops/.venv/lib/python3.14/site-packages/click/core.py", line 1912, in invoke
    return _process_result(sub_ctx.command.invoke(sub_ctx))
                           ~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^
  File "/home/zelitosaide/Documents/mlops/.venv/lib/python3.14/site-packages/click/core.py", line 1308, in invoke
    return ctx.invoke(self.callback, **ctx.params)
           ~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/zelitosaide/Documents/mlops/.venv/lib/python3.14/site-packages/click/core.py", line 877, in invoke
    return callback(*args, **kwargs)
  File "/home/zelitosaide/Documents/mlops/.venv/lib/python3.14/site-packages/mlflow/models/cli.py", line 309, in build_docker
    python_api.build_docker(**kwargs)
    ~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^
  File "/home/zelitosaide/Documents/mlops/.venv/lib/python3.14/site-packages/mlflow/models/python_api.py", line 83, in build_docker
    get_flavor_backend(model_uri, docker_build=True, env_manager=env_manager).build_image(
    ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^
        model_uri,
        ^^^^^^^^^^
    ...<4 lines>...
        base_image=base_image,
        ^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/home/zelitosaide/Documents/mlops/.venv/lib/python3.14/site-packages/mlflow/pyfunc/backend.py", line 390, in build_image
    docker_utils.build_image_from_context(context_dir=cwd, image_name=image_name)
    ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/zelitosaide/Documents/mlops/.venv/lib/python3.14/site-packages/mlflow/models/docker_utils.py", line 228, in build_image_from_context
    raise RuntimeError("Docker build failed.")
RuntimeError: Docker build failed.

In [ ]:
# To fix I ran this:
mlflow models build-docker --model-uri models:/m-72fd213672a142f78e0d562f8751006a --name "house_sales_model" --env-manager local

sudo .venv/bin/mlflow models build-docker --model-uri models:/m-72fd213672a142f78e0d562f8751006a --name "house_sales_model" --env-manager local

In [ ]:
docker run -p 5001:8080 "house_sales_model"